In [2]:
from dotenv import load_dotenv
import os

load_dotenv()

google_key = os.getenv("GOOGLE_API_KEY")

In [26]:
from youtube_transcript_api import YouTubeTranscriptApi, TranscriptsDisabled
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_google_genai import ChatGoogleGenerativeAI, GoogleGenerativeAIEmbeddings
from langchain_community.vectorstores import FAISS
from langchain_core.prompts import PromptTemplate
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_core.runnables import RunnableParallel, RunnablePassthrough, RunnableLambda
from langchain_core.output_parsers import StrOutputParser

step 1a -> Indexing(Docment Ingestion)


In [4]:
video_id = "Gfr50f6ZBvo"

try:
    ytt_api = YouTubeTranscriptApi()
    transcript_list = ytt_api.fetch(video_id)

    # flatten it to a plain text
    transcript = " ".join(chunk.text for chunk in transcript_list)

except TranscriptsDisabled:
    print("No captions available for this video")

step 1b -> Indexing(Text Splitting)


In [5]:
splitter = RecursiveCharacterTextSplitter(
    chunk_size=1000,
    chunk_overlap=200
)
chunks = splitter.create_documents([transcript])

In [ ]:
embedding_model = HuggingFaceEmbeddings(model="all-MiniLM-L6-v2")

vector_store = FAISS.from_documents(
    documents=chunks,
    embedding = embedding_model
)

In [ ]:
vector_store.index_to_docstore_id

In [ ]:
vector_store.get_by_ids(["a0a8c22f-e4c0-480e-8f8b-e90dd420dd0f"])

step 2 -> Retrieval


In [9]:
retriever = vector_store.as_retriever(search_type="similarity", search_kwargs={"k":4})

In [ ]:
# retriever.invoke("what is deepmind")

step 3 -> Augmentation


In [15]:
prompt = PromptTemplate(
    template="""
        You are a helpful assistant.
        Answer only from the provided transcript context.
        If the context is insufficient, just say you don't know.

        {context}
        question:{question}
""",
    input_variables=["context", "question"]
)

In [ ]:
# question = "is the topic of aliens discussed in this video? If yes then what was discussed?"
# retrieved_docs = retriever.invoke(question)

In [ ]:
# context_text = "\n\n".join(doc.page_content for doc in retrieved_docs)

In [ ]:
# final_prompt = prompt.invoke({"context": context_text, "question": question})

step 4 -> Generation


In [22]:
llm = ChatGoogleGenerativeAI(model="gemini-2.5-flash")

In [ ]:
# ans = llm.invoke(final_prompt)
# print(ans.content)

Improvements -> Building a chain


In [3]:
def format_docs(retrieved_docs):
    context_text = "\n\n".join(doc.page_content for doc in retrieved_docs)
    return context_text

In [ ]:
parallel_chain = RunnableParallel({
    "context": retriever | RunnableLambda(format_docs),
    "question": RunnablePassthrough()
})

In [ ]:
# parallel_chain.invoke("what is Demis?")

In [35]:
parser = StrOutputParser()

In [ ]:
main_chain = parallel_chain | prompt | llm | parser

In [ ]:
result = main_chain.invoke("is the topic of aliens discussed in this video? If yes then what was discussed?")
print(result)